# Worked Capstone: Customer Segmentation with Stability and Profiling

**Domain:** Unsupervised learning  
**Primary dataset:** `customer_segments.csv`  
**Level:** Practitioner to Advanced

## Business goal

Develop candidate customer segments that are stable, interpretable, and connected to a real action rather than treating clusters as natural truth.

This is a worked reference project. First attempt the corresponding phase project independently; then use this capstone to compare framing, evaluation, code structure, and communication.

## Decision questions

        1. What representation defines similarity?
2. How many clusters are stable?
3. How do algorithms differ?
4. Are profiles actionable?

        ## Definition of done

        - [ ] Scaled feature representation
- [ ] K comparison
- [ ] Seed stability
- [ ] GMM comparison
- [ ] Profiles
- [ ] 2D PCA view
- [ ] Limitations

## End-to-end workflow

```text
Decision and scope
      ↓
Data contract and quality
      ↓
Exploration and hypotheses
      ↓
Baseline and evaluation design
      ↓
Candidate method(s)
      ↓
Held-out / temporal evaluation
      ↓
Error, slice, and sensitivity analysis
      ↓
Artifacts, limitations, recommendation
```

At every stage, distinguish calculation correctness, statistical validity, operational validity, and decision validity.

## Risk register

        | Risk | Mitigation |
        |---|---|
        | Scale determines clusters | Standardize or justify units. |
| k chosen from one diagnostic | Combine stability, separation, and usefulness. |
| Profiles become stereotypes | Use aggregates cautiously and validate intended action. |

In [ ]:
from pathlib import Path
import sys
import json
import warnings
warnings.filterwarnings("ignore")

_candidates = [Path.cwd(), *Path.cwd().parents]
COURSE_ROOT = next((p for p in _candidates if (p / "datasets").exists()), Path.cwd())
DATA_DIR = COURSE_ROOT / "datasets"
ARTIFACT_DIR = COURSE_ROOT / "artifacts"
ARTIFACT_DIR.mkdir(exist_ok=True)
sys.path.insert(0, str(COURSE_ROOT))

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import display

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
print(f"Course root: {COURSE_ROOT}")

## 1. Representation

Exclude identifiers and standardize because income and satisfaction use different units.

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score,adjusted_rand_score
from sklearn.mixture import GaussianMixture
from sklearn.decomposition import PCA

df=pd.read_csv(DATA_DIR/"customer_segments.csv")
features=df.drop(columns="customer_id")
scaler=StandardScaler()
X=scaler.fit_transform(features)
display(features.describe().T)

## 2. Select candidate k

Compare inertia and silhouette across k, then consider interpretability and stability.

In [ ]:
rows=[]
for k in range(2,9):
    model=KMeans(n_clusters=k,n_init=25,random_state=42).fit(X)
    rows.append({"k":k,"inertia":model.inertia_,"silhouette":silhouette_score(X,model.labels_)})
diagnostics=pd.DataFrame(rows)
display(diagnostics)
chosen_k=int(diagnostics.sort_values("silhouette",ascending=False).iloc[0].k)
print("Diagnostic candidate k:",chosen_k)

## 3. Stability across seeds and bootstrap samples

Random initialization should not fundamentally change the result for a stable solution.

In [ ]:
label_sets=[]
for seed in [1,2,3,4,42]:
    label_sets.append(KMeans(n_clusters=chosen_k,n_init=25,random_state=seed).fit_predict(X))
stability=[adjusted_rand_score(label_sets[0],labels) for labels in label_sets[1:]]
print("Adjusted Rand stability:",stability)
labels=label_sets[-1]

## 4. Profiles and soft memberships

Profiles describe observed averages; they do not define individual identity or causal response.

In [ ]:
profile=features.assign(cluster=labels).groupby("cluster").agg(["mean","median","count"])
display(profile.round(2))
gmm=GaussianMixture(n_components=chosen_k,random_state=42).fit(X)
soft=gmm.predict_proba(X)
print("Mean maximum GMM membership confidence:",soft.max(axis=1).mean())
print("Ambiguous members (<0.65 max responsibility):",int((soft.max(axis=1)<.65).sum()))

## 5. PCA visualization and artifact

Use PCA only to visualize a projection; cluster validation remains in the full representation.

In [ ]:
embedding=PCA(n_components=2,random_state=42).fit_transform(X)
fig,ax=plt.subplots(figsize=(6,5))
scatter=ax.scatter(embedding[:,0],embedding[:,1],c=labels,s=18)
ax.set(title="Candidate segments in a two-component PCA projection",xlabel="PC1",ylabel="PC2")
fig.colorbar(scatter,ax=ax); plt.show()

output=df.copy()
output["cluster"]=labels
output["membership_confidence"]=soft.max(axis=1)
output.to_csv(ARTIFACT_DIR/"capstone_customer_segments.csv",index=False)

## Model/project card

Complete this before presenting the result:

| Field | Statement |
|---|---|
| Intended use | |
| Excluded use | |
| Data population and coverage | |
| Target/metric definition | |
| Evaluation split | |
| Baseline | |
| Primary result | |
| Known limitations | |
| Important subgroup behaviour | |
| Human review / abstention | |
| Monitoring | |
| Owner and review cadence | |

## Final reflection

1. Which result changed your initial belief?
2. Which assumption creates the largest residual risk?
3. What simpler alternative was competitive?
4. What evidence is still required before an operational decision?
5. What would you monitor first after release?

Re-run the notebook from a clean kernel and verify generated artifacts before considering the capstone complete.